In [1]:
import os

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from typing import Literal

In [3]:
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this tool when you need to perform calculations.
    
    Args:
        expression: A math expression like '2 + 2' or '100 * 0.15'
    """
    try:
        result = eval(expression)           # Evaluate the math expression
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

In [4]:
tools = [calculator]

In [5]:
from langchain_core.messages import SystemMessage
llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)

In [6]:
llm_with_tools = llm.bind_tools(tools)

In [7]:
tools_by_name = {t.name: t for t in tools}

In [8]:
print (tools_by_name)

{'calculator': StructuredTool(name='calculator', description="Evaluates a mathematical expression and returns the result.\nUse this tool when you need to perform calculations.\n\nArgs:\n    expression: A math expression like '2 + 2' or '100 * 0.15'", args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x74e78962b9c0>)}


In [9]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


In [10]:
def research_handler(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

In [11]:
def tool_node(state: State) -> dict:
    results = []
    for tc in state["messages"][-1].tool_calls:
        result = tools_by_name[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return {"messages": results}

In [12]:
def should_continue(state: State) -> Literal["tools", "__end__"]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return "__end__"

In [13]:
builder = StateGraph(State)

In [14]:
       
builder.add_node("research", research_handler)

In [15]:
builder.add_node("tools",tool_node )

In [16]:
builder.add_edge(START, "research")

In [17]:
builder.add_conditional_edges("research", should_continue, ["tools", "__end__"])

In [18]:

builder.add_edge("tools", "research")

In [19]:
router = builder.compile() 

In [20]:
out = router.invoke({"messages": [HumanMessage(content="multiply 2 by 3?")]})

In [21]:
print(out["messages"][-1].content)

The result of multiplying 2 by 3 is 6.


In [22]:
out = router.invoke({"messages": [HumanMessage(content="calculate 30*5")]})

In [23]:
print(out["messages"][-1].content)

The result of \( 30 \times 5 \) is 150.


In [24]:
print(out["messages"])

[HumanMessage(content='calculate 30*5', additional_kwargs={}, response_metadata={}, id='b57763e2-93aa-4e30-a6f1-b9f3161cbbb1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 92, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 2.34e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.34e-05, 'upstream_inference_prompt_cost': 1.38e-05, 'upstream_inference_completions_cost': 9.6e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_6ec6bfb92d', 'id': 'gen-1786013285-grgQZXeB4cFhgh4DTtHU', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd6af-fe19-7423-81d9-f8391cbcd492-0', to